# Email Header Anomaly Detection — Exploratory Analysis

This notebook provides an interactive walkthrough of the extended pipeline:

1. Load processed features
2. Explore feature distributions (ham vs spam)
3. Compute and visualise feature importance
4. Train models, inspect confidence scores
5. Calibration curves and reliability diagrams
6. Summary comparison across feature groups

**Prerequisite**: run `scripts/generate_synthetic_demo.py` first to populate `data/processed/`.

In [ ]:
import sys, warnings
from pathlib import Path

warnings.filterwarnings('ignore')
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
%matplotlib inline

with open(ROOT / 'config.yaml') as f:
    CFG = yaml.safe_load(f)

print('Ready.')

## 1. Load Data

In [ ]:
# Load whichever dataset exists (real or synthetic)
for candidate in ['datasetA_processed.csv', 'synthetic_A.csv']:
    path = ROOT / 'data' / 'processed' / candidate
    if path.exists():
        df_A = pd.read_csv(path)
        print(f'Loaded Dataset A: {path.name}  shape={df_A.shape}')
        break
else:
    raise FileNotFoundError('No processed Dataset A found. Run generate_synthetic_demo.py first.')

print(df_A['label'].value_counts().rename({0: 'ham', 1: 'spam'}).to_string())
df_A.head(3)

## 2. Feature Distributions (Ham vs Spam)

In [ ]:
from src.features.feature_groups import get_feature_subset, describe_groups

print('Feature groups:')
for g, desc in describe_groups().items():
    cols = get_feature_subset(df_A, g)
    print(f'  {g}: {len(cols):2d} features — {desc}')

In [ ]:
# Visualise mean feature values per class for G2 (domain matching)
g2_cols = get_feature_subset(df_A, 'G2')
means = df_A.groupby('label')[g2_cols].mean().T.rename(columns={0: 'Ham', 1: 'Spam'})

fig, ax = plt.subplots(figsize=(10, 5))
means.plot(kind='barh', ax=ax)
ax.set_title('G2 (Domain Matching) — Mean Feature Value by Class')
ax.set_xlabel('Mean Value')
plt.tight_layout()
plt.show()

In [ ]:
# G5: authentication results distribution
g5_cols = get_feature_subset(df_A, 'G5')
means_g5 = df_A.groupby('label')[g5_cols].mean().T.rename(columns={0: 'Ham', 1: 'Spam'})

fig, ax = plt.subplots(figsize=(9, 4))
means_g5.plot(kind='barh', ax=ax)
ax.set_title('G5 (Authentication) — Mean Feature Value by Class')
plt.tight_layout()
plt.show()

## 3. Feature Importance

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from src.features.feature_importance import compute_permutation_importance, plot_importance

label_col = 'label'
feature_cols = [c for c in df_A.columns if c != label_col]
X = df_A[feature_cols]
y = df_A[label_col]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
rf.fit(X_tr, y_tr)
print(f'RF test accuracy: {rf.score(X_te, y_te):.4f}')

In [ ]:
df_imp = compute_permutation_importance(rf, X_te, y_te, n_repeats=5)
plot_importance(df_imp, top_n=20, title='Permutation Importance (RF, Dataset A)')

## 4. Confidence Scores

In [ ]:
from src.models.confidence import get_confidence_scores, compute_calibration_metrics

y_prob = get_confidence_scores(rf, X_te)
calib = compute_calibration_metrics(y_te.values, y_prob)
print('Calibration metrics (RF, all features):')
for k, v in calib.items():
    print(f'  {k}: {v:.4f}')

In [ ]:
# Confidence distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ham_probs  = y_prob[y_te.values == 0]
spam_probs = y_prob[y_te.values == 1]

axes[0].hist(ham_probs,  bins=30, alpha=0.7, label='Ham',  color='steelblue')
axes[0].hist(spam_probs, bins=30, alpha=0.7, label='Spam', color='salmon')
axes[0].set_title('Predicted Confidence Distribution (RF)')
axes[0].set_xlabel('P(spam)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Reliability diagram
from src.utils.plotting import plot_calibration_curve
plot_calibration_curve(y_te.values, y_prob, model_name='RF', feature_group='All', ax=axes[1])

plt.tight_layout()
plt.show()

## 5. Multi-model Confidence Comparison

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from src.models.confidence import get_confidence_scores, compute_calibration_metrics

# Use G6 features
from sklearn.preprocessing import StandardScaler

g6_cols = get_feature_subset(df_A, 'G6')
X6 = df_A[g6_cols]
X6_tr, X6_te, y6_tr, y6_te = train_test_split(X6, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X6_tr_s = scaler.fit_transform(X6_tr)
X6_te_s = scaler.transform(X6_te)

models_to_compare = {
    'RF':  RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'MLP': MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=300, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
}

results = []
fig, axes = plt.subplots(1, len(models_to_compare), figsize=(15, 4))

for ax, (name, model) in zip(axes, models_to_compare.items()):
    model.fit(X6_tr_s, y6_tr)
    prob = get_confidence_scores(model, X6_te_s)
    calib = compute_calibration_metrics(y6_te.values, prob)
    acc = model.score(X6_te_s, y6_te)
    results.append({'Model': name, 'Accuracy': acc, **calib})
    plot_calibration_curve(y6_te.values, prob, model_name=name, feature_group='G6', ax=ax)

plt.suptitle('Reliability Diagrams — G6 Features', fontsize=13)
plt.tight_layout()
plt.show()

pd.DataFrame(results).set_index('Model').round(4)

## 6. Load Experiment Results (if available)

In [ ]:
results_path = ROOT / 'results' / 'feature_combinations' / 'synthetic_comparison.csv'

if results_path.exists():
    df_exp = pd.read_csv(results_path)
    print(f'Loaded {len(df_exp)} experiment rows.')
    display(df_exp[['dataset', 'model_name', 'feature_group', 'accuracy', 'f1',
                     'brier_score', 'ece', 'train_time_s', 'inference_ms_per_1k']].round(4))
else:
    print(f'No results found at {results_path}.')
    print('Run: python scripts/generate_synthetic_demo.py')

In [ ]:
if results_path.exists() and 'df_exp' in dir():
    from src.utils.plotting import plot_accuracy_vs_time, plot_metric_heatmap
    
    for ds in df_exp['dataset'].unique():
        sub = df_exp[df_exp['dataset'] == ds]
        plot_accuracy_vs_time(sub, dataset_label=ds)
        plt.show()
        
        plot_metric_heatmap(sub, metric_col='f1', title=f'F1 Heatmap — Dataset {ds}')
        plt.show()

## 7. SVM Calibration Walkthrough

In [ ]:
from sklearn.svm import SVC
from src.models.confidence import calibrate_svm

svm = SVC(C=1.0, kernel='rbf', probability=False, random_state=42)
svm.fit(X6_tr_s, y6_tr)

calibrated_svm = calibrate_svm(svm, X6_tr_s, y6_tr, method='sigmoid', cv=5)
prob_svm = get_confidence_scores(calibrated_svm, X6_te_s)
calib_svm = compute_calibration_metrics(y6_te.values, prob_svm)

print('SVM (Platt-calibrated) metrics:')
for k, v in calib_svm.items():
    print(f'  {k}: {v:.4f}')

fig, ax = plt.subplots(figsize=(6, 5))
plot_calibration_curve(y6_te.values, prob_svm, model_name='SVM (Platt)', feature_group='G6', ax=ax)
plt.tight_layout()
plt.show()

## 8. Feature Group Size vs Performance

In [ ]:
if results_path.exists() and 'df_exp' in dir():
    ds_A = df_exp[(df_exp['dataset'] == 'A') & (df_exp['model_name'] == 'RF')].copy()
    if 'n_features' not in ds_A.columns:
        ds_A['n_features'] = 0
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    axes[0].scatter(ds_A['n_features'], ds_A['accuracy'], s=100)
    for _, row in ds_A.iterrows():
        axes[0].annotate(row['feature_group'],
                         (row['n_features'], row['accuracy']),
                         xytext=(3, 3), textcoords='offset points', fontsize=9)
    axes[0].set_xlabel('Number of Features')
    axes[0].set_ylabel('Accuracy')
    axes[0].set_title('RF: Feature Count vs Accuracy (Dataset A)')
    
    axes[1].scatter(ds_A['inference_ms_per_1k'], ds_A['accuracy'], s=100)
    for _, row in ds_A.iterrows():
        axes[1].annotate(row['feature_group'],
                         (row['inference_ms_per_1k'], row['accuracy']),
                         xytext=(3, 3), textcoords='offset points', fontsize=9)
    axes[1].set_xlabel('Inference Time (ms/1k)')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('RF: Inference Time vs Accuracy (Dataset A)')
    
    plt.tight_layout()
    plt.show()
else:
    print('Run the synthetic demo first to generate results.')